# YOLOv11s vs YOLOv12m — Model Comparison on Merged 2-Class Dataset

**Dissertation Phase 5 · Evaluation & Model Selection**

This notebook compares two YOLO architectures trained on the **same merged
2-class dataset** (Phase 4 dataset) under identical conditions:

| | YOLOv11s | YOLOv12m |
|---|---|---|
| **Architecture class** | Small | Medium |
| **Dataset** | Merged 2-class | Merged 2-class |
| **Image size** | 640 px | 640 px |
| **Hardware** | NVIDIA L4 | NVIDIA L4 |
| **Epochs** | 148 (best @ 118) | 105 (early stop, patience=10) |

> **Note on fairness:** The comparison is between different model size classes
> (Small vs Medium). A true apples-to-apples comparison would require
> YOLOv12s, which was not available due to time constraints. Despite this,
> the comparison is still valuable for justifying the final model selection
> based on the accuracy-vs-efficiency trade-off.


## 1. Setup


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

matplotlib.rcParams.update({
    'figure.facecolor': '#0e0e1a',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#3a3a5c',
    'text.color':       '#e0e0e0',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#9ba8b5',
    'ytick.color':      '#9ba8b5',
    'grid.color':       '#2d2d42',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})

COLOR_V11 = '#FF7814'   # orange  -- YOLOv11s
COLOR_V12 = '#4CC9F0'   # cyan    -- YOLOv12m


## 2. Model Results

Values taken directly from validation output printed at end of each training run.


In [ ]:
# YOLOv11s  -- Phase 4 best model (148 epochs, best @ epoch 118, 1.782 h)
v11 = dict(
    name           = 'YOLOv11s',
    size_class     = 'Small',
    epochs_trained = 148,
    best_epoch     = 118,
    train_hours    = 1.782,
    params_M       = 9.41,
    gflops         = 21.3,
    map50          = 0.662,
    map50_cig      = 0.784,
    map50_like     = 0.541,
    precision      = 0.825,
    recall         = 0.612,
)
v11['f1'] = 2 * v11['precision'] * v11['recall'] / (v11['precision'] + v11['recall'])

# YOLOv12m  -- Phase 5 (105 epochs, early stopped patience=10, 6.208 h)
# Source: Yolo12m.ipynb training output (saved before Colab disconnection)
v12 = dict(
    name           = 'YOLOv12m',
    size_class     = 'Medium',
    epochs_trained = 105,
    best_epoch     = 95,   # estimated: early stop fires ~10 epochs after best
    train_hours    = 6.208,
    params_M       = 20.1,
    gflops         = 67.1,
    map50          = 0.745,
    map50_cig      = 0.845,
    map50_like     = 0.645,
    precision      = 0.754,
    recall         = 0.693,
)
v12['f1'] = 2 * v12['precision'] * v12['recall'] / (v12['precision'] + v12['recall'])

models = [v11, v12]
colors = [COLOR_V11, COLOR_V12]
names  = [m['name'] for m in models]

print('Models loaded:')
for m in models:
    print(f"  {m['name']:10s}  mAP@50={m['map50']:.1%}  P={m['precision']:.1%}  R={m['recall']:.1%}  F1={m['f1']:.1%}")


## 3. Overall Validation Metrics

mAP@50, Precision, Recall, and F1 on the merged 2-class validation set.

YOLOv12m achieves higher accuracy across all metrics, which is expected
given it is a larger model (Medium vs Small).


In [ ]:
metrics = ['map50', 'precision', 'recall', 'f1']
labels  = ['mAP@50', 'Precision', 'Recall', 'F1']

x = np.arange(len(labels))
w = 0.32
fig, ax = plt.subplots(figsize=(9, 5))

for i, (model, color) in enumerate(zip(models, colors)):
    vals = [model[m] for m in metrics]
    bars = ax.bar(x + i*w, vals, width=w, color=color, alpha=0.85,
                  label=f"{model['name']} ({model['size_class']})", zorder=3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.1%}', ha='center', va='bottom', fontsize=8.5, color=color)

ax.set_xticks(x + w/2)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Overall Validation Metrics -- Merged 2-Class Dataset')
ax.legend()
ax.grid(axis='y', zorder=0)
plt.tight_layout()
import os; os.makedirs('notebooks/figures', exist_ok=True)
plt.savefig('notebooks/figures/05_overall_metrics.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Per-Class mAP@50

YOLOv12m leads on both classes. The `cigarette_like_object` class (pens,
straws) remains harder for both models — it benefits more from the extra
capacity of the Medium architecture.


In [ ]:
classes = ['Cigarette', 'Cig-like Object']
keys    = ['map50_cig', 'map50_like']

x = np.arange(len(classes))
w = 0.32
fig, ax = plt.subplots(figsize=(7, 5))

for i, (model, color) in enumerate(zip(models, colors)):
    vals = [model[k] for k in keys]
    bars = ax.bar(x + i*w, vals, width=w, color=color, alpha=0.85,
                  label=f"{model['name']} ({model['size_class']})", zorder=3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.1%}', ha='center', va='bottom', fontsize=9, color=color)

ax.set_xticks(x + w/2)
ax.set_xticklabels(classes)
ax.set_ylim(0, 1.05)
ax.set_ylabel('mAP@50')
ax.set_title('Per-Class mAP@50 -- Merged 2-Class Dataset')
ax.legend()
ax.grid(axis='y', zorder=0)
plt.tight_layout()
plt.savefig('notebooks/figures/05_per_class_map.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Model Efficiency — Parameters & GFLOPs

YOLOv12m has **2.1x more parameters** and **3.1x higher computational cost**
than YOLOv11s. For a real-time smoking detection system running on a laptop
or embedded device, this is a significant disadvantage.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))

for ax, key, ylabel, title in zip(
    axes,
    ['params_M', 'gflops'],
    ['Parameters (M)', 'GFLOPs'],
    ['Model Parameters', 'Computational Cost (GFLOPs)'],
):
    vals = [m[key] for m in models]
    bars = ax.bar([f"{m['name']}\n({m['size_class']})" for m in models],
                  vals, color=colors, alpha=0.85, zorder=3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f'{val:.1f}', ha='center', va='bottom', fontsize=10, color='#e0e0e0')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis='y', zorder=0)

# Annotate the ratio
ratio_params = v12['params_M'] / v11['params_M']
ratio_flops  = v12['gflops']  / v11['gflops']
axes[0].set_title(f'Parameters (YOLOv12m is {ratio_params:.1f}x larger)')
axes[1].set_title(f'GFLOPs (YOLOv12m is {ratio_flops:.1f}x heavier)')
plt.suptitle('Architecture Efficiency Comparison', y=1.02)
plt.tight_layout()
plt.savefig('notebooks/figures/05_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Training Cost

YOLOv12m required **3.5x more training time** despite converging earlier
(early stopped at epoch 105 vs 148). The larger attention-based architecture
is inherently slower to train per epoch.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
vals = [m['train_hours'] for m in models]
bars = ax.bar([f"{m['name']}\n({m['size_class']})" for m in models],
              vals, color=colors, alpha=0.85, zorder=3)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.3f} h', ha='center', va='bottom', fontsize=10, color='#e0e0e0')
ratio = v12['train_hours'] / v11['train_hours']
ax.set_ylabel('Training time (hours)')
ax.set_title(f'Training Time (YOLOv12m took {ratio:.1f}x longer)')
ax.grid(axis='y', zorder=0)
plt.tight_layout()
plt.savefig('notebooks/figures/05_training_time.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Accuracy vs Efficiency Trade-off

The ideal model sits in the **top-left** corner: high accuracy, low compute.
This plot shows why YOLOv11s is the better choice for real-time deployment.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for model, color in zip(models, colors):
    ax.scatter(model['gflops'], model['map50'],
               s=model['params_M'] * 15,   # bubble size = model size
               color=color, alpha=0.85, zorder=3,
               label=f"{model['name']} ({model['size_class']})")
    ax.annotate(
        f"  {model['name']}\n  mAP={model['map50']:.1%}, {model['gflops']} GFLOPs",
        (model['gflops'], model['map50']),
        fontsize=8.5, color=color,
    )

ax.set_xlabel('GFLOPs (lower = faster inference)')
ax.set_ylabel('mAP@50 (higher = more accurate)')
ax.set_title('Accuracy vs Efficiency Trade-off\n(bubble size = number of parameters)')
ax.legend()
ax.grid(zorder=0)
# Annotate ideal zone
ax.annotate('Ideal zone', xy=(15, 0.9), fontsize=9, color='#52c052',
            arrowprops=dict(arrowstyle='->', color='#52c052'),
            xytext=(30, 0.88))
plt.tight_layout()
plt.savefig('notebooks/figures/05_accuracy_vs_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Model Selection Justification

### Why YOLOv11s was selected for the final system

Despite YOLOv12m achieving higher accuracy metrics (mAP@50: 74.5% vs 66.2%),
**YOLOv11s was selected** as the production model for the following reasons:

#### 8.1 Comparison Caveat
The comparison is between **different model size classes** (Small vs Medium).
YOLOv12m has 2.1x more parameters and 3.1x higher GFLOPs. A fair comparison
would require YOLOv12s (Small), which was not trained due to time constraints.
Given the same size class, YOLOv11s and YOLOv12s are expected to perform
comparably based on Phase 1 results (YOLOv11s: 72.1%, YOLOv12s: 70.6% mAP@50).

#### 8.2 Real-Time Deployment Requirement
The dissertation system targets **real-time camera inference** on standard
hardware (laptop CPU/GPU). YOLOv11s at 21.3 GFLOPs is substantially more
suitable than YOLOv12m at 67.1 GFLOPs for this use case.

#### 8.3 Precision vs Recall Trade-off
YOLOv11s achieves **higher Precision (82.5% vs 75.4%)**, meaning fewer false
alarms — important for a detection system where false positives cause
unnecessary disruption. YOLOv12m trades precision for recall.

#### 8.4 Training Efficiency
YOLOv11s trained in **1.782 hours** vs 6.208 hours for YOLOv12m — a 3.5x
reduction, making iterative improvement and hyperparameter tuning practical.

#### 8.5 Model Size
YOLOv11s weights: ~18 MB · YOLOv12m weights: ~40.7 MB.
Smaller model size is important for deployment on resource-constrained systems.

### Summary Decision Table

| Criterion | YOLOv11s | YOLOv12m | Winner |
|-----------|----------|----------|--------|
| mAP@50 | 66.2% | 74.5% | YOLOv12m |
| Precision | **82.5%** | 75.4% | **YOLOv11s** |
| F1 Score | 71.1% | **72.2%** | YOLOv12m |
| Parameters | **9.41 M** | 20.1 M | **YOLOv11s** |
| GFLOPs | **21.3** | 67.1 | **YOLOv11s** |
| Training time | **1.782 h** | 6.208 h | **YOLOv11s** |
| Model file size | **~18 MB** | ~40.7 MB | **YOLOv11s** |
| Real-time viability | **High** | Medium | **YOLOv11s** |

> **Conclusion:** YOLOv11s provides the optimal balance of detection accuracy
> and computational efficiency for a real-time smoking detection system.
> The accuracy gap is partially explained by the size class difference.
> YOLOv11s is the recommended model for deployment.


## 9. Full Numeric Summary


In [ ]:
try:
    import pandas as pd
    rows = []
    for m in models:
        rows.append({
            'Model':         m['name'],
            'Size Class':    m['size_class'],
            'Params (M)':    m['params_M'],
            'GFLOPs':        m['gflops'],
            'Epochs':        f"{m['epochs_trained']} (best: {m['best_epoch']})",
            'Train (h)':     f"{m['train_hours']:.3f}",
            'mAP@50':        f"{m['map50']:.1%}",
            'mAP@50 Cig':    f"{m['map50_cig']:.1%}",
            'mAP@50 Like':   f"{m['map50_like']:.1%}",
            'Precision':     f"{m['precision']:.1%}",
            'Recall':        f"{m['recall']:.1%}",
            'F1':            f"{m['f1']:.1%}",
        })
    df = pd.DataFrame(rows).set_index('Model')
    display(df.T)
except ImportError:
    for m in models:
        print(m)
